# Current ranker HMM integration

Use the **current 32-feature direction ranker**, not the earlier 24-feature ranker. Reuse the verified Step 8 transition, Viterbi and forward/backward functions unchanged.

Exclude trips `00000033` and `00000054` from decoder tuning, preserve the saved trip folds, and select one complete training trip per fold based only on its size. Tune beta and emission weight from current OOF probabilities only. The five-trip grid score is a development score, not independent test performance.

Student-test quality labels never enter decoder inputs. After configuration is frozen, compare classifier-only and decoded predictions against the same reference labels, including candidate misses. This notebook records evidence only; it does not promote a model or change the registry.

In [1]:
%run 08_hmm_helpers.ipynb
from datetime import datetime
from zoneinfo import ZoneInfo

model_folder = Path("results/direction_features")
output = Path("results/hmm/direction_ranker")
output.mkdir(parents=True, exist_ok=True)
contract = json.loads(Path("outputs/folds/contract.json").read_text())
summary = pd.read_csv("outputs/candidates/summary.csv", dtype={"trip_id": str})
folds = pd.read_csv("outputs/folds/trip_folds.csv", dtype={"trip_id": str})
summary = summary.merge(folds, on="trip_id", validate="one_to_one")
excluded = json.loads(Path("outputs/training_exclusions.json").read_text())["excluded_trip_ids"]
source_config = json.loads((model_folder / "config.json").read_text())
assert set(source_config["excluded_trip_ids"]) == set(excluded)
summary = summary[~summary["trip_id"].isin(excluded)].copy()
assert len(json.loads((model_folder / "feature_columns.json").read_text())) == 32
summary["size_difference"] = (summary["gps_points"] - 700).abs()
sample = summary.sort_values(["size_difference", "trip_id"]).groupby("fold").first().reset_index()
train_points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
train_points = train_points[train_points["trip_id"].isin(sample["trip_id"])].merge(
    pd.read_csv(contract["fold_file"], usecols=["group_id", "fold"]), on="group_id", validate="one_to_one")
ids = set(train_points["group_id"])
oof_parts = []
for chunk in pd.read_csv(model_folder / "oof_xgb_ranker.csv", chunksize=250000):
    oof_parts.append(chunk[chunk["group_id"].isin(ids)])
oof = pd.concat(oof_parts, ignore_index=True)
assert (oof["fold"] == oof["group_id"].map(train_points.set_index("group_id")["fold"])).all()
assert not set(sample["trip_id"]) & set(excluded)
sample.to_csv(output / "tuning_trips.csv", index=False)
print("Decoder tuning trips:", sample["trip_id"].tolist())

Decoder tuning trips: ['00000027', '00000037', '00000079', '00000045', '00000029']


## Tune on training OOF only

Try a 3×3 grid. Each trip is decoded independently, then correct selected segments are counted over its valid labelled points. Show per-fold metrics, then choose by pooled Top-1. Deterministic ties prefer smaller transition and emission weights.

In [2]:
if (output / "hmm_config.json").exists():
    config = json.loads((output / "hmm_config.json").read_text())
    assert config["tuning_source"] == str(model_folder / "oof_xgb_ranker.csv")
    assert config["tuning_trips"] == sample["trip_id"].tolist()
    assert config["tuning_source_bytes"] == (model_folder / "oof_xgb_ranker.csv").stat().st_size
    assert config["tuning_source_mtime_ns"] == (model_folder / "oof_xgb_ranker.csv").stat().st_mtime_ns
    assert (output / "tuning_grid.csv").exists() and (output / "tuning_per_fold.csv").exists()
    print("Reusing frozen OOF-only decoder configuration:", config)
else:
    prepared = {}
    for row in sample.itertuples(index=False):
        point_rows = train_points[train_points["trip_id"] == row.trip_id]
        scores = oof[oof["group_id"].isin(point_rows["group_id"])]
        network = load_network(row.trip_id)
        prepared[row.trip_id] = prepare_trip(scores, point_rows, network)
        print("Prepared training trip:", row.trip_id, flush=True)
    
    records = []
    for beta in [0.25, 1.0, 4.0]:
        for emission_weight in [0.5, 1.0, 2.0]:
            for row in sample.itertuples(index=False):
                decoded, _ = decode_trip(prepared[row.trip_id], beta, emission_weight, posterior=False)
                valid = train_points[(train_points["trip_id"] == row.trip_id) & ~train_points["bad_gps"]]
                joined = valid.merge(decoded[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
                correct = joined["true_arc_id"].eq(joined["segment_id"]).sum()
                records.append({"beta": beta, "emission_weight": emission_weight, "trip_id": row.trip_id,
                    "fold": row.fold, "points": len(valid), "correct": correct, "top1": correct / len(valid)})
            print("Tuned beta", beta, "emission weight", emission_weight, flush=True)
    trials = pd.DataFrame(records)
    trials.to_csv(output / "tuning_per_fold.csv", index=False)
    results = trials.groupby(["beta", "emission_weight"], as_index=False)[["points", "correct"]].sum()
    results["top1"] = results["correct"] / results["points"]
    results.to_csv(output / "tuning_grid.csv", index=False)
    best = results.sort_values(["top1", "beta", "emission_weight"], ascending=[False, True, True]).iloc[0]
    config = {"beta": float(best["beta"]), "emission_weight": float(best["emission_weight"]),
        "temperature": 1.0, "gap_reset_s": 30, "transition_distance_scale_floor_m": 20, "unreachable_penalty": 20,
        "transition_cost_cap": 100, "topology": "directed reachability through full graph",
        "transition_distance": "endpoint-bridge lower-bound surrogate; not shortest road distance",
        "model_source": str(model_folder), "feature_count": 32, "excluded_trip_ids": excluded,
        "tuning_source": str(model_folder / "oof_xgb_ranker.csv"),
        "tuning_source_bytes": (model_folder / "oof_xgb_ranker.csv").stat().st_size,
        "tuning_source_mtime_ns": (model_folder / "oof_xgb_ranker.csv").stat().st_mtime_ns,
        "tuning_trips": sample["trip_id"].tolist(), "tuning_folds": sample["fold"].tolist(),
        "tuning_points": int(best["points"]), "tuning_top1": float(best["top1"]),
        "posterior": "forward-backward marginals; Viterbi joint path reported separately",
        "selected_at": datetime.now(ZoneInfo("Asia/Kolkata")).isoformat()}
    (output / "hmm_config.json").write_text(json.dumps(config, indent=2) + "\n")
    display(results.sort_values("top1", ascending=False))
    del prepared


Prepared training trip: 00000027


Prepared training trip: 00000037


Prepared training trip: 00000079


Prepared training trip: 00000045


Prepared training trip: 00000029


Tuned beta 0.25 emission weight 0.5


... (5 outputs left out)

Tuned beta 4.0 emission weight 0.5


Tuned beta 4.0 emission weight 1.0


Tuned beta 4.0 emission weight 2.0


,beta,emission_weight,points,correct,top1
5,1.00,2.0,3433,3210,0.935042
2,0.25,2.0,3433,3195,0.930673
4,1.00,1.0,3433,3173,0.924264
1,0.25,1.0,3433,3168,0.922808
8,4.00,2.0,3433,3154,0.918730
3,1.00,0.5,3433,3110,0.905913
0,0.25,0.5,3433,3085,0.898631
7,4.00,1.0,3433,3026,0.881445
6,4.00,0.5,3433,2916,0.849403


In [3]:
valid = train_points[~train_points["bad_gps"]].copy()
ranker_scores = oof.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
winners = ranker_scores.loc[ranker_scores.groupby("group_id")["score"].idxmax(), ["group_id", "segment_id"]]
joined = valid.merge(winners, on="group_id", how="left", validate="one_to_one")
joined["correct"] = joined["true_arc_id"].eq(joined["segment_id"])
baseline = joined.groupby(["trip_id", "fold"], as_index=False).agg(points=("group_id", "size"), correct=("correct", "sum"))
baseline["top1"] = baseline["correct"] / baseline["points"]
baseline.to_csv(output / "tuning_baseline.csv", index=False)
ranker_top1 = float(joined["correct"].sum() / len(valid))
tuning_comparison = {"points": len(valid), "ranker_only_top1": ranker_top1,
    "hmm_top1": config["tuning_top1"],
    "gain_percentage_points": 100 * (config["tuning_top1"] - ranker_top1),
    "note": "Same five whole OOF tuning trips and valid-point denominator; development score, not independent test."}
(output / "tuning_comparison.json").write_text(json.dumps(tuning_comparison, indent=2) + "\n")
print("Same-sample decoder development comparison:", tuning_comparison)
del oof, ranker_scores

Same-sample decoder development comparison: {'points': 3433, 'ranker_only_top1': 0.9321293329449462, 'hmm_top1': 0.935042237110399, 'gain_percentage_points': 0.29129041654528676, 'note': 'Same five whole OOF tuning trips and valid-point denominator; development score, not independent test.'}


## Decode student-test emissions

Read model probabilities for **all** observations, including flagged ones, then decode without inspecting their reference labels. Fill empty-shortlist observations with `segment_id = -1`. Quality labels are used only later by notebook 09.

Save both the Viterbi path and the candidate marginal posteriors. Decoder-only timing excludes feature generation and base-model prediction; final evaluation must add the latter.

In [4]:
scores = pd.read_csv(model_folder / "student_test_predictions.csv")
points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str},
    usecols=["trip_id", "group_id", "point_id", "time", "lon", "lat"])
selected_parts, probability_parts, timings = [], [], []
for trip_id, point_rows in points.groupby("trip_id", sort=False):
    start = perf_counter()
    network = load_network(trip_id, "student_test")
    emissions = scores[scores["group_id"].isin(point_rows["group_id"])]
    prepared = prepare_trip(emissions, point_rows, network, config["gap_reset_s"])
    decoded, posterior = decode_trip(prepared, config["beta"], config["emission_weight"])
    elapsed = perf_counter() - start
    complete = point_rows[["trip_id", "group_id", "point_id"]].merge(
        decoded[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
    complete["segment_id"] = complete["segment_id"].fillna(-1).astype("int64")
    selected_parts.append(complete)
    if posterior is not None:
        probability_parts.append(posterior)
    timings.append({"trip_id": trip_id, "points": len(point_rows), "decoder_s": elapsed})
    print("Decoded student-test trip:", trip_id, "seconds", round(elapsed, 2), flush=True)
selected = pd.concat(selected_parts, ignore_index=True)
probabilities = pd.concat(probability_parts, ignore_index=True).sort_values(["group_id", "segment_id"])
assert selected["group_id"].is_unique and len(selected) == len(points)
np.testing.assert_allclose(probabilities.groupby("group_id")["score"].sum(), 1, atol=1e-6)
selected.to_csv(output / "decoded_points.csv", index=False)
probabilities.to_parquet(output / "candidate_probabilities.parquet", index=False)
pd.DataFrame(timings).to_csv(output / "timings.csv", index=False)
(output / "timings.json").write_text(json.dumps({"decoder_predict_s": sum(r["decoder_s"] for r in timings),
    "timing_scope": "network loading/projection, directed topology, transition preparation, Viterbi and forward-backward",
    "includes": "network loading/projection, directed topology, transition preparation, Viterbi and forward-backward",
    "excludes": "feature generation and base model inference"}, indent=2) + "\n")
print("Saved Viterbi rows:", len(selected), "| Posterior rows:", len(probabilities))

Decoded student-test trip: 00000000 seconds 6.94


Decoded student-test trip: 00000007 seconds 3.58


Decoded student-test trip: 00000009 seconds 6.05


Decoded student-test trip: 00000040 seconds 7.79


Decoded student-test trip: 00000052 seconds 0.96


Decoded student-test trip: 00000056 seconds 16.4


Decoded student-test trip: 00000065 seconds 2.5


Decoded student-test trip: 00000076 seconds 5.92


Decoded student-test trip: 00000077 seconds 13.89


Decoded student-test trip: 00000091 seconds 4.58


Saved Viterbi rows: 19160 | Posterior rows: 1186138


## Current classifier-only versus decoder evaluation

Viterbi selections determine Top-1, macro F1 and the confusion matrix. Decoder log loss uses forward/backward marginal probabilities, not the Viterbi states or the unchanged emission scores.

Continuity counts directed-unreachable transitions through the full graph; legal multi-edge paths are supported. It does not establish shortest-path distance or speed feasibility. Prediction timing adds base inference to measured decoder loading, topology, transition, Viterbi and posterior time; feature generation is excluded. No student-test score changes the frozen decoder settings.

In [5]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
valid_points = points[~points["bad_gps"]].copy()
emissions = pd.read_csv(model_folder / "student_test_predictions.csv")
posteriors = pd.read_parquet(output / "candidate_probabilities.parquet")
decoded = pd.read_csv(output / "decoded_points.csv", dtype={"trip_id": str})
assert len(decoded) == len(points) and decoded["group_id"].is_unique
pd.testing.assert_frame_equal(
    emissions[["group_id", "segment_id"]].sort_values(["group_id", "segment_id"]).reset_index(drop=True),
    posteriors[["group_id", "segment_id"]].sort_values(["group_id", "segment_id"]).reset_index(drop=True),
    check_dtype=False,
)
assert config["tuning_source_bytes"] == (model_folder / "oof_xgb_ranker.csv").stat().st_size
assert config["tuning_source_mtime_ns"] == (model_folder / "oof_xgb_ranker.csv").stat().st_mtime_ns

In [6]:
def evaluate_predictions(probabilities, selected):
    probabilities = probabilities[["group_id", "segment_id", "score"]].copy()
    assert np.isfinite(probabilities["score"]).all() and probabilities["score"].between(0, 1).all()
    np.testing.assert_allclose(probabilities.groupby("group_id")["score"].sum(), 1, atol=1e-6)
    assert not selected.duplicated("group_id").any()
    selected = selected[selected["segment_id"] >= 0].copy()
    keys = probabilities[["group_id", "segment_id"]]
    assert len(selected.merge(keys, on=["group_id", "segment_id"])) == len(selected)
    rows = probabilities.merge(valid_points[["group_id", "true_arc_id"]], on="group_id", validate="many_to_one")
    rows["label"] = rows["segment_id"].eq(rows["true_arc_id"]).astype("int8")
    chosen = selected[["group_id", "segment_id"]].assign(predicted=1)
    rows = rows.merge(chosen, on=["group_id", "segment_id"], how="left", validate="one_to_one")
    prediction = rows["predicted"].fillna(0).astype("int8")
    precision, recall, f1, _ = precision_recall_fscore_support(
        rows["label"], prediction, labels=[0, 1], zero_division=0)
    positive_scores = rows.loc[rows["label"] == 1, "score"]
    predicted = valid_points.merge(selected[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
    correct = predicted["segment_id"].eq(predicted["true_arc_id"])
    assert rows.groupby("group_id")["label"].sum().le(1).all()
    return {"top1": float(correct.sum() / len(valid_points)), "macro_f1": float(f1.mean()),
            "precision": float(precision[1]), "recall": float(recall[1]),
            "log_loss": float(-np.log(positive_scores.clip(lower=1e-15)).mean()),
            "candidate_coverage": len(positive_scores) / len(valid_points),
            "valid_points": len(valid_points), "candidate_misses": len(valid_points) - len(positive_scores),
            "confusion_matrix": confusion_matrix(rows["label"], prediction, labels=[0, 1]).tolist()}


def top_candidates(probabilities):
    ordered = probabilities.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
    return ordered.loc[ordered.groupby("group_id")["score"].idxmax(), ["group_id", "segment_id"]].reset_index(drop=True)

In [7]:
def route_continuity(selected):
    selected = selected[["group_id", "segment_id"]].copy()
    merged = points[["trip_id", "group_id", "point_id"]].merge(selected, on="group_id", how="left", validate="one_to_one")
    counts = {"transition_pairs": 0, "reachable_pairs": 0, "disconnected_jumps": 0, "missing_transition_pairs": 0}
    for trip_id, frame in merged.groupby("trip_id", sort=True):
        frame = frame.sort_values("point_id")
        network = load_network(trip_id, partition="student_test")
        roads = frame["segment_id"].to_numpy()
        for previous, current in zip(roads[:-1], roads[1:]):
            if not np.isfinite(previous) or not np.isfinite(current) or min(previous, current) < 0:
                counts["missing_transition_pairs"] += 1
                continue
            counts["transition_pairs"] += 1
            reachable = transition_is_reachable(network, int(previous), int(current))
            counts["reachable_pairs"] += int(reachable)
            counts["disconnected_jumps"] += int(not reachable)
    counts["route_continuity"] = counts["reachable_pairs"] / counts["transition_pairs"] if counts["transition_pairs"] else None
    return counts

In [8]:
base_metrics = json.loads((model_folder / "student_test_metrics.json").read_text())
base_seconds = float(base_metrics["prediction_s"])
decode_seconds = json.loads((output / "timings.json").read_text())["decoder_predict_s"]
selections = {"direction_ranker": top_candidates(emissions), "direction_ranker_hmm": decoded[["group_id", "segment_id"]]}
probabilities = {"direction_ranker": emissions, "direction_ranker_hmm": posteriors}
records = []
for name in ["direction_ranker", "direction_ranker_hmm"]:
    metrics = evaluate_predictions(probabilities[name], selections[name])
    continuity = route_continuity(selections[name])
    seconds = base_seconds + (decode_seconds if name.endswith("hmm") else 0)
    records.append({"model": name, **metrics, **continuity, "prediction_s": seconds,
        "timing_scope": "saved base inference plus optional decoder; excludes feature generation"})
comparison = pd.DataFrame(records)
assert abs(comparison.iloc[0]["top1"] - base_metrics["top1"]) < 1e-12
assert comparison.iloc[0]["valid_points"] == base_metrics["points"]
comparison.to_csv(output / "comparison.csv", index=False)
(output / "evaluation_status.json").write_text(json.dumps({
    "model_source": str(model_folder), "feature_count": 32, "selection_performed": False,
    "decoder_config_frozen_from_training_oof": True, "tuning_exclusions": excluded,
    "root_integration_note": "No promotion; use this current-source comparison when integrating the registry."}, indent=2) + "\n")
display(comparison[["model", "top1", "macro_f1", "log_loss", "disconnected_jumps", "prediction_s"]])
print("Current HMM beta:", config["beta"], "| Emission weight:", config["emission_weight"])
print("Top-1 change (percentage points):", 100 * (comparison.iloc[1]["top1"] - comparison.iloc[0]["top1"]))

,model,top1,macro_f1,log_loss,disconnected_jumps,prediction_s
0,direction_ranker,0.898904,0.948622,0.276072,11,0.368653
1,direction_ranker_hmm,0.891754,0.944988,0.632145,0,68.973861


Current HMM beta: 1.0 | Emission weight: 2.0
Top-1 change (percentage points): -0.7150313152400889
